In [ ]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [ ]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS) # can only add reasoning effort with the responses api

qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # re write this to use instructor
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        response_model=RAG_GenerationResponse,
        reasoning={"effort": "none"}
    )
    return response

def rag_pipeline(question, topk_k=5):
    retrievedContext = retrieve_data(question, k=topk_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [ ]:
output = rag_pipeline("Do you have any earphones")

In [ ]:
output["answer"]

### RAG Pipline with grounding context (the items used to create the answer)

In [ ]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of item used to answer the question")
    description: str = Field(description="Description of item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the the")

In [ ]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS) # can only add reasoning effort with the responses api

qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # re write this to use instructor
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        response_model=RAGGenerationResponse,
        reasoning={"effort": "none"}
    )
    return response

def rag_pipeline(question, top_k=5):
    retrievedContext = retrieve_data(question, k=top_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [ ]:
res = rag_pipeline("Do you have any headphones", top_k = 15) # so we increaser the top k to 10 but it only used 5 of them as refernecse from the 10 retrieved context ids

In [ ]:
res['references']

In [ ]:
print(res['answer'])

In [ ]:
print(res['retrieved_context_ids'])

In [ ]:
res